In [ ]:
##__Sentiment Analysis using NLP & ML__##

End-to-end pipeline with preprocessing, feature engineering, and model comparison.

import pandas as pd
import numpy as np
import re
import string

import nltk
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score
     
##__Load Dataset__##

Replace with your dataset file.

df = pd.read_csv('dataset.csv')  # change file name
df.head()

print(df.shape)
print(df['sentiment'].value_counts())
     
##__Preprocessing__##

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    words = [lemmatizer.lemmatize(w) for w in words]
    return " ".join(words)

df['clean_text'] = df['review'].apply(clean_text)
df[['review','clean_text']].head()
     
##__Feature Engineering__##

bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_text'])

tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_text'])

y = df['sentiment']
     
##__Train Test Split__##

X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)
     
##__Models__##

# Logistic Regression
lr = LogisticRegression(max_iter=200)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)
y_pred_nb = nb.predict(X_test_bow)

# Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train_bow, y_train)
y_pred_dt = dt.predict(X_test_bow)
     
##__Evaluation__##

def evaluate(y_test, y_pred, name):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

evaluate(y_test, y_pred_lr, "Logistic Regression")
evaluate(y_test, y_pred_nb, "Naive Bayes")
evaluate(y_test, y_pred_dt, "Decision Tree")
     
##__Conclusion__##

Logistic Regression usually performs best with TF-IDF
Naive Bayes is fast and efficient
Decision Tree may overfit
Best Model: Logistic Regression
Best Feature: TF-IDF